# Notebook 1: Simulating Supply Chain Costs

**Goal:** Understand how costs arise in a multi-echelon supply chain under the
newsvendor cost model, and how holding/backorder cost parameters affect total
system cost.

We will:
1. Generate realistic semiconductor demand
2. Simulate a 4-echelon supply chain
3. Decompose costs into holding vs. backorder at each echelon
4. Examine how service level shifts the cost balance
5. Visualize everything with publication-grade plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

SEED = 966  # Saudi country code

## 1. Generate Demand

The demand model is an AR(1) process with seasonal variation and a structural
shock (e.g., an AI-driven demand surge at week 104).

In [ ]:
from deepbullwhip import SemiconductorDemandGenerator
from deepbullwhip.diagnostics.plots import plot_demand_trajectory

gen = SemiconductorDemandGenerator()
demand = gen.generate(T=156, seed=SEED)

fig = plot_demand_trajectory(demand, shock_period=104)
plt.show()

print(f"Mean weekly demand: {demand.mean():.2f} units")
print(f"Std deviation:      {demand.std():.2f} units")

## 2. The Newsvendor Cost Model

At each echelon, per-period cost is:

$$C(I_t) = \begin{cases}
h \cdot I_t & \text{if } I_t \geq 0 \quad \text{(holding cost — excess inventory)} \\
b \cdot |I_t| & \text{if } I_t < 0 \quad \text{(backorder cost — unmet demand)}
\end{cases}$$

Each echelon has **different** cost parameters, reflecting that downstream
echelons (closer to the customer) face higher penalty for stockouts:

In [ ]:
from deepbullwhip import default_semiconductor_config

configs = default_semiconductor_config()

print(f"{'Echelon':<14} {'Lead Time':>10} {'h (hold)':>10} {'b (back)':>10} {'b/h ratio':>10}")
print("-" * 58)
for cfg in configs:
    print(f"{cfg.name:<14} {cfg.lead_time:>10d} {cfg.holding_cost:>10.2f} "
          f"{cfg.backorder_cost:>10.2f} {cfg.backorder_cost/cfg.holding_cost:>10.1f}")

## 3. Simulate & Inspect Costs

In [ ]:
from deepbullwhip import SerialSupplyChain

chain = SerialSupplyChain()
fm = np.full_like(demand, demand.mean())
fs = np.full_like(demand, demand.std())
result = chain.simulate(demand, fm, fs)

print(f"{'Echelon':<14} {'Fill Rate':>10} {'Holding':>12} {'Backorder':>12} {'Total':>12}")
print("-" * 64)
for er in result.echelon_results:
    inv = er.inventory_levels
    h_cost = float(er.costs[inv >= 0].sum())
    b_cost = float(er.costs[inv < 0].sum())
    print(f"{er.name:<14} {er.fill_rate:>10.1%} {h_cost:>12,.1f} {b_cost:>12,.1f} {er.total_cost:>12,.1f}")
print("-" * 64)
print(f"{'TOTAL':<14} {'':>10} {'':>12} {'':>12} {result.total_cost:>12,.1f}")

## 4. Cost Time Series

Visualize how costs evolve over time at each echelon. Green bars = holding cost
(too much inventory), gold bars = backorder cost (stockout).

In [ ]:
from deepbullwhip.diagnostics.plots import plot_cost_timeseries

fig = plot_cost_timeseries(result)
plt.show()

## 5. Echelon Deep Dive: Foundry (E3)

The foundry has the longest lead time (12 weeks), making it most vulnerable to
demand variability. Let's inspect its orders, inventory, and costs in detail.

In [ ]:
from deepbullwhip.diagnostics.plots import plot_echelon_detail

fig = plot_echelon_detail(demand, result, echelon_index=2)
plt.show()

## 6. Inventory Dynamics

Inventory on-hand shows where stockouts (backorders) happen. The shaded regions
below zero directly correspond to backorder costs.

In [ ]:
from deepbullwhip.diagnostics.plots import plot_inventory_levels

fig = plot_inventory_levels(result)
plt.show()

## 7. Service Level vs. Cost Tradeoff

Higher service level means more safety stock (higher holding cost) but fewer
stockouts (lower backorder cost). Let's sweep service levels and visualize the tradeoff.

In [ ]:
from deepbullwhip import EchelonConfig
from deepbullwhip.diagnostics.plots import plot_cost_decomposition

service_levels = [0.70, 0.80, 0.90, 0.95, 0.99]
results_by_sl = {}

for sl in service_levels:
    cfgs = [
        EchelonConfig(c.name, c.lead_time, c.holding_cost, c.backorder_cost,
                      c.depreciation_rate, service_level=sl)
        for c in default_semiconductor_config()
    ]
    ch = SerialSupplyChain.from_config(cfgs)
    results_by_sl[f"SL={sl:.0%}"] = ch.simulate(demand, fm, fs)

fig = plot_cost_decomposition(results_by_sl)
plt.show()

In [ ]:
# Tabulate the tradeoff
import pandas as pd

rows = []
for name, res in results_by_sl.items():
    h_total = sum(float(er.costs[er.inventory_levels >= 0].sum()) for er in res.echelon_results)
    b_total = sum(float(er.costs[er.inventory_levels < 0].sum()) for er in res.echelon_results)
    avg_fr = np.mean([er.fill_rate for er in res.echelon_results])
    rows.append({"Service Level": name, "Holding": h_total, "Backorder": b_total,
                 "Total": res.total_cost, "Avg Fill Rate": f"{avg_fr:.1%}"})

pd.DataFrame(rows).set_index("Service Level")

## 8. Summary Dashboard

In [ ]:
from deepbullwhip.diagnostics.plots import plot_summary_dashboard

fig = plot_summary_dashboard(demand, result)
plt.show()

## Key Takeaways

1. **Costs increase upstream** — longer lead times amplify order variability,
   causing larger inventory swings and higher costs
2. **Holding vs. backorder tradeoff** — higher service level reduces stockouts
   but increases safety stock; the optimal SL depends on the h/b ratio
3. **The Foundry (E3)** with L=12 weeks is the most expensive echelon despite
   having the lowest per-unit cost rates, because its order variability is highest